### `file_access_events`
File/data system.

| Field | Type | Notes |
|---|---|---|
| event_id | UUID (PK) | |
| system_identifier | string | user_id |
| resource_id | string (FK → resources) | |
| access_timestamp | timestamp | |
| action | string | read / write / download |
| data_volume_mb | float | Enables both count-based and volume-based anomaly detection (many-small vs few-large) |

In [0]:
pip install pycountry

In [0]:
import uuid
import datetime
import random
import pycountry
import pytz
import zoneinfo
from pyspark.sql.functions import col, collect_set

In [0]:
# List of 44 european countries
europe_countries = [
    "Albania", "Andorra", "Austria", "Belarus", "Belgium", 
    "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia", 
    "Denmark", "Estonia", "Finland", "France", "Germany", 
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", 
    "Latvia", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", 
    "Moldova", "Monaco", "Montenegro", "Netherlands", "North Macedonia", 
    "Norway", "Poland", "Portugal", "Romania", "Russia", 
    "San Marino", "Serbia", "Slovakia", "Slovenia", "Spain", 
    "Sweden", "Switzerland", "Ukraine", "United Kingdom"
]

# Actions specs 
action_list = ["read", "write", "download"]
action_weights = [0.75, 0.20, 0.05]

In [0]:
entity_df = spark.read.table("entity_risk_platform.seed_data.entities").select("entity_id", "entity_type", "activity_pattern", "tier", "home_country")

file_identifier_df = spark.read.table("entity_risk_platform.seed_data.entity_system_identifiers").filter(col("system_name") == "file_access")

grant_df = spark.read.table("entity_risk_platform.seed_data.access_grants").groupBy("entity_id").agg(collect_set("resource_id").alias("resources"))

# grant_df.show()

file_entity_df = file_identifier_df.join(entity_df, "entity_id")

# file_entity_df.show()

file_entities = [row.asDict() for row in file_entity_df.collect()]

entity_resources = {row["entity_id"]: row["resources"] for row in grant_df.collect()}

# print(file_entities)
# print(entity_resources)


Resource: randomly chosen from the entity's actual access_grants

Frequency: Human weekday 10-15, weekend 0-5. Service_account: scheduled fixed 5, always_on 5-10, triggered 1-5. 
Agent: Fully-autonomous 10-15, Semi-autonomous 5-10, Supervised 0-5

Timing: Human — same working-hours window as login (home timezone). Non-human — random throughout the day

Action: read 75% / write 20% / download 5%

data_volume_mb: read = 0, write = uniform(1, 50), download = uniform(50, 100)

In [0]:
europe_timzones = {}

for ec in europe_countries:
    country = pycountry.countries.search_fuzzy(ec)[0]
    country_code = country.alpha_2
    country_timezone = pytz.country_timezones.get(country_code)[0]
    europe_timzones[ec] = country_timezone

# print(europe_timzones)

In [0]:
# chosen_entity = random.choice(file_entities)
# print(chosen_entity)

In [0]:
def generate_file_event_count(entity, date):
    event_count = 0
    if(entity["entity_type"] == "human"):
        weekno = date.weekday()
        if weekno < 5:
            event_count = random.randrange(10, 16)
        else:
            event_count = random.randrange(0, 6)
    elif(entity["entity_type"] == "service_account"):
        if(entity["activity_pattern"] == "scheduled"):
            event_count = 5
        elif(entity["activity_pattern"] == "triggered"):
            event_count = random.randrange(1, 6)
        elif(entity["activity_pattern"] == "always_on"):
            event_count = random.randrange(5, 11)
    elif(entity["entity_type"] == "agent"):
        if(entity["tier"] == "Fully-autonomous"):
            event_count = random.randrange(10, 16)
        elif(entity["tier"] == "Semi-autonomous"):
            event_count = random.randrange(5, 11)
        elif(entity["tier"] == "Supervised"):
            event_count = random.randrange(0, 6)

    return event_count

In [0]:
def generate_timestamp(date, timezone, min_sec, max_sec):
    random_date = datetime.datetime.combine(date.date(), datetime.time.min) + datetime.timedelta(seconds=random.randrange(min_sec, max_sec))
    local_date = random_date.replace(tzinfo=zoneinfo.ZoneInfo(timezone))
    utc_date = local_date.astimezone(zoneinfo.ZoneInfo("UTC"))
    return utc_date

# print(generate_timestamp(datetime.datetime.now(), chosen_entity["home_country"], 0, 3600))

In [0]:
def generate_access_timestamp(date, entity):
    if(entity["entity_type"] == "human"):
        # 8 AM
        min_seconds = 8 * 3600 
        # 6 PM
        max_seconds = 18 * 3600
        return generate_timestamp(date, europe_timzones[entity["home_country"]], min_seconds, max_seconds)
    else:
        # 12 AM
        min_seconds = 0
        # 12 PM
        max_seconds = 24 * 3600
        return generate_timestamp(date, "UTC", min_seconds, max_seconds)

In [0]:
def generate_file_activity():
    action = random.choices(action_list, weights=action_weights)[0]
    if(action == "write"):
        volume = random.uniform(1, 50)
    elif(action == "download"):
        volume = random.uniform(50, 100)
    else:
        volume = 0
    return action, volume

In [0]:
target_date = datetime.datetime.now()

In [0]:
file_event_list = []

In [0]:
def generate_file_events(date, entities):
    events = []
    for chosen_entity in entities:
        # Skip entities that don't have any access grants
        resources = entity_resources.get(chosen_entity["entity_id"])
        if not resources:
            continue
            
        file_events = generate_file_event_count(chosen_entity, date)
        for _ in range(file_events):
            action, volume = generate_file_activity()
            events.append({
                "event_id": str(uuid.uuid4()),
                "system_identifier": chosen_entity["system_identifier"],
                "resource_id": random.choice(resources),
                "access_timestamp": generate_access_timestamp(date, chosen_entity),
                "action": action,
                "data_volume_mb": volume
            })
    return events

file_event_list = generate_file_events(target_date, file_entities)
print(file_event_list)